<a href="https://colab.research.google.com/github/VictoriaUsman/Big-Data/blob/main/CSV%20to%20MongoDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pymongo import MongoClient
import pandas as pd

hostname = "t4sfj8.h.filess.io"
database = "Mongo_NOSQL_peacegreen"
port = "61003"
username = "Mongo_NOSQL_peacegreen"
password = "6f8c10c062905cf3bbe4e06fa035a206b1b1eb07"

url = f"mongodb://{username}:{password}@{hostname}:{port}/{database}?authSource=admin"

client = MongoClient(url)
db = client[database]

In [2]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 34.0 MB/s eta 0:00:00


In [6]:
print(db)

Database(MongoClient(host=['t4sfj8.h.filess.io:61003'], document_class=dict, tz_aware=False, connect=True, authsource='admin'), 'Mongo_NOSQL_peacegreen')


# Task
Read the 'product_category_name_translation.csv' file into a pandas DataFrame, prepare it for insertion into MongoDB, upload the data to a new collection in the connected MongoDB database, and verify the upload by querying the collection.

## Read CSV file

### Subtask:
Read the 'product_category_name_translation.csv' file into a pandas DataFrame.


**Reasoning**:
To read the 'product_category_name_translation.csv' file into a pandas DataFrame named `df_translation`, I will use the `pd.read_csv()` function and then display the first few rows to verify the data has been loaded correctly.



In [7]:
df_translation = pd.read_csv('product_category_name_translation.csv')
print("First 5 rows of the DataFrame:")
print(df_translation.head())

First 5 rows of the DataFrame:
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


## Prepare data for MongoDB

### Subtask:
Convert the DataFrame into a format suitable for MongoDB insertion (e.g., a list of dictionaries).


**Reasoning**:
To prepare the data for MongoDB insertion, I need to convert the `df_translation` DataFrame into a list of dictionaries. Each dictionary will represent a document to be inserted into the MongoDB collection. The `to_dict('records')` method is ideal for this purpose.



In [8]:
data_for_mongo = df_translation.to_dict('records')
print("First 5 documents for MongoDB insertion:")
print(data_for_mongo[:5])

First 5 documents for MongoDB insertion:
[{'product_category_name': 'beleza_saude', 'product_category_name_english': 'health_beauty'}, {'product_category_name': 'informatica_acessorios', 'product_category_name_english': 'computers_accessories'}, {'product_category_name': 'automotivo', 'product_category_name_english': 'auto'}, {'product_category_name': 'cama_mesa_banho', 'product_category_name_english': 'bed_bath_table'}, {'product_category_name': 'moveis_decoracao', 'product_category_name_english': 'furniture_decor'}]


## Upload to MongoDB

### Subtask:
Insert the prepared data into a new collection in the connected MongoDB database.


**Reasoning**:
To insert the prepared data into MongoDB, I will define a collection name, access it through the 'db' object, and then use the 'insert_many()' method with 'data_for_mongo', finally printing the 'inserted_ids' for verification.



In [9]:
collection_name = 'product_category_translations'
collection = db[collection_name]

# Insert the data into the collection
result = collection.insert_many(data_for_mongo)

print(f"Successfully inserted {len(result.inserted_ids)} documents into the '{collection_name}' collection.")
print("First 5 inserted IDs:")
print(result.inserted_ids[:5])

OperationFailure: Authentication failed., full error: {'ok': 0.0, 'errmsg': 'Authentication failed.', 'code': 18, 'codeName': 'AuthenticationFailed'}

**Reasoning**:
The previous code failed because of an `OperationFailure: Authentication failed` error. This usually indicates that the `authSource` in the MongoDB connection string is incorrect. Given that the username and the database name are the same (`Mongo_NOSQL_peacegreen`), it's highly probable that the user is defined within its own database, not `admin`. I will correct the `authSource` parameter in the connection URL to match the database name.



In [10]:
from pymongo import MongoClient
import pandas as pd

hostname = "t4sfj8.h.filess.io"
database = "Mongo_NOSQL_peacegreen"
port = "61003"
username = "Mongo_NOSQL_peacegreen"
password = "6f8c10c062905cf3bbe4e06fa035a206b1b1eb07"

# Corrected authSource to match the database name
url = f"mongodb://{username}:{password}@{hostname}:{port}/{database}?authSource={database}"

client = MongoClient(url)
db = client[database]
print("MongoDB connection client and database object initialized with corrected authSource.")

MongoDB connection client and database object initialized with corrected authSource.


**Reasoning**:
With the MongoDB connection now successfully re-established using the corrected authentication source, I will re-attempt to insert the prepared `data_for_mongo` into the specified collection. This should now proceed without the previous authentication error.



In [11]:
collection_name = 'product_category_translations'
collection = db[collection_name]

# Insert the data into the collection
result = collection.insert_many(data_for_mongo)

print(f"Successfully inserted {len(result.inserted_ids)} documents into the '{collection_name}' collection.")
print("First 5 inserted IDs:")
print(result.inserted_ids[:5])

Successfully inserted 71 documents into the 'product_category_translations' collection.
First 5 inserted IDs:
[ObjectId('695b8410a485ee359fcf2563'), ObjectId('695b8410a485ee359fcf2564'), ObjectId('695b8410a485ee359fcf2565'), ObjectId('695b8410a485ee359fcf2566'), ObjectId('695b8410a485ee359fcf2567')]


## Verify Upload to MongoDB

### Subtask:
Verify the upload by querying the collection.

**Reasoning**:
To verify the data upload, I will query the `product_category_translations` collection to retrieve a few documents. This will confirm that the insertion was successful and the data is stored as expected. I will use `collection.find_one()` to retrieve a single document and `collection.count_documents({})` to check the total number of documents, ensuring it matches the original DataFrame's length.

**Reasoning**:
To verify the data upload, I will query the MongoDB collection to retrieve a single document and count the total number of documents. This confirms the insertion was successful and the data is stored as expected.



In [12]:
collection_name = 'product_category_translations'
collection = db[collection_name]

# Query for one document to verify content
one_document = collection.find_one()
print("One document from the collection:")
print(one_document)

# Count the total number of documents
document_count = collection.count_documents({})
print(f"Total number of documents in '{collection_name}' collection: {document_count}")

# Verify if the count matches the original DataFrame length (assuming df_translation was used)
expected_count = len(df_translation)
print(f"Expected number of documents (from df_translation): {expected_count}")

if document_count == expected_count:
    print("Document count matches the original DataFrame length. Verification successful!")
else:
    print("Document count does NOT match the original DataFrame length. Verification failed.")

One document from the collection:
{'_id': ObjectId('695b8410a485ee359fcf2563'), 'product_category_name': 'beleza_saude', 'product_category_name_english': 'health_beauty'}
Total number of documents in 'product_category_translations' collection: 71
Expected number of documents (from df_translation): 71
Document count matches the original DataFrame length. Verification successful!


## Summary:

### Data Analysis Key Findings
*   The `product_category_name_translation.csv` file, containing 71 records, was successfully read into a pandas DataFrame with columns `product_category_name` and `product_category_name_english`.
*   The DataFrame was converted into a list of 71 dictionaries, which is the required format for MongoDB insertion.
*   An initial attempt to connect and upload data to MongoDB failed due to an `Authentication failed` error, which was resolved by correcting the `authSource` parameter in the MongoDB connection string to match the database name.
*   After correcting the authentication issue, all 71 documents were successfully inserted into the `product_category_translations` collection in the MongoDB database.
*   Verification confirmed that the total number of documents in the MongoDB collection (71) exactly matched the number of records from the original DataFrame, ensuring a complete and successful data upload.

### Insights or Next Steps
*   Ensure that MongoDB connection parameters, especially `authSource`, are correctly specified to prevent authentication failures during data operations.
*   This translated product category data can now be used for further analysis, reporting, or integration with other datasets within the MongoDB database, particularly for applications requiring English category names.
